# Deteccion de lavado de dinero en remesas (PaySim + IBM AML)

Sistema de dos etapas: un autoencoder secuencial con atencion que aprende el
comportamiento normal de remesas (Etapa A), y un clasificador que hace transfer
learning desde ese encoder para distinguir lavado de dinero de comportamiento
legitimo (Etapa B). Notebook autocontenido, reproducible en Colab con GPU T4
gratuita en menos de 30 minutos.

## 1. Configuracion y entorno

In [ ]:
import sys, os, time, json, random, math, warnings, glob, copy
NOTEBOOK_START = time.time()

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print(f"torch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU: ninguna (CPU) -- el entrenamiento sera mas lento")


In [ ]:
# Configuracion global. Todos los hiperparametros y flags viven aqui:
# nada de constantes magicas repartidas por el notebook.
CFG = dict(
    SEQUENCE_SOURCE="ibm_aml",     # "ibm_aml" | "paysim" -- ver diagnostico en seccion 2
    REBUILD_FROM_RAW=False,        # True = pipeline completo desde CSV crudo de Kaggle
    PROCESSED_URL="TODO_RELEASE_ASSET_URL",  # TODO: URL de un release con el .npz ya procesado
    MAX_LEN=32,                    # valor inicial; se recalcula con el percentil 90 en la seccion 3
    MIN_LEN=5,
    N_SENDERS=60_000,              # subconjunto de remitentes; ver justificacion en seccion 3
    HIDDEN=64,
    LATENT=64,
    BATCH=256,
    EPOCHS_A=15,
    EPOCHS_B=12,                   # subido de 10: con FREEZE_EPOCHS mas corto, deja mas epocas de fine-tuning real
    FREEZE_EPOCHS=1,               # bajado de 3: menos tiempo "desperdiciado" en fase congelada
    LR_HEAD=1e-3,
    LR_ENCODER=3e-4,               # subido de 1e-4: permite al encoder alejarse mas rapido de una inicializacion de Etapa A que resulto poco informativa (ver seccion 6)
    FOCAL_GAMMA=2.0,
    FOCAL_ALPHA=0.25,
    ALERT_RATE=0.01,               # capacidad operativa del equipo de cumplimiento
    SEEDS=[0, 1, 2],
    DEVICE="cuda" if torch.cuda.is_available() else "cpu",
)
CFG


> **Decision:** usar IBM AML como fuente principal de secuencias por remitente
> (`SEQUENCE_SOURCE = "ibm_aml"`), con PaySim como validacion secundaria.
>
> **Justificacion:** se confirma con el diagnostico de la seccion 2 -- en PaySim
> `nameOrig` es casi unico por fila, lo que hace imposible construir secuencias
> temporales por remitente con mas de una o dos transacciones.
>
> **Alternativa descartada:** usar PaySim como fuente principal (como sugiere el
> enunciado al llamarlo "dataset principal"). Se descarta porque el objetivo del
> Componente 1 es explicitamente representar el comportamiento *a lo largo del
> tiempo* de un remitente, algo que PaySim no permite construir de forma fiable.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
set_seed(CFG["SEEDS"][0])
try:
    torch.use_deterministic_algorithms(True)
    print("Determinismo total activado.")
except Exception as e:
    print(f"No se pudo forzar determinismo total (se continua sin el): {e}")


class Timer:
    """Mide y reporta el tiempo de un bloque; los bloques pesados deben reportarse."""
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        print(f"[TIEMPO] {self.label}: {time.time() - self.t0:.1f}s")


## 2. Carga de datos y diagnostico de la fuente

In [ ]:
# Esquema normalizado comun a ambos datasets:
# sender_id, timestamp, amount, tx_type, dest_id, label

def load_ibm_aml(path):
    df = pd.read_csv(path)
    rename_map = {
        "Timestamp": "timestamp",
        "Account": "sender_id",
        "Account.1": "dest_id",
        "Amount Paid": "amount",
        "Payment Format": "tx_type",
        "Is Laundering": "label",
    }
    df = df.rename(columns=rename_map)
    missing = [c for c in ["timestamp", "sender_id", "dest_id", "amount", "tx_type", "label"] if c not in df.columns]
    if missing:
        raise KeyError(f"Columnas esperadas de IBM AML no encontradas: {missing}. "
                        f"Columnas disponibles: {list(df.columns)}")
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df[["sender_id", "timestamp", "amount", "tx_type", "dest_id", "label"]].copy()


def load_paysim(path):
    df = pd.read_csv(path)
    df = df.rename(columns={
        "nameOrig": "sender_id",
        "nameDest": "dest_id",
        "type": "tx_type",
        "isFraud": "label",
    })
    # PaySim no trae timestamp real, solo "step" (horas desde el inicio de la simulacion).
    df["timestamp"] = pd.to_datetime(df["step"], unit="h", origin="2023-01-01")
    return df[["sender_id", "timestamp", "amount", "tx_type", "dest_id", "label"]].copy()


IBM_AML_FILE = "HI-Small_Trans.csv"      # variante pequeña; ver justificacion de N_SENDERS en seccion 3


def get_raw_data_path(dataset):
    """IBM AML: descarga (o reutiliza cache) solo HI-Small_Trans.csv, no las
    ~7.6GB del dataset completo (seis variantes HI/LI x Small/Medium/Large).
    PaySim: descarga el dataset completo -- solo trae un CSV, sin ambiguedad,
    y la descarga selectiva por archivo demostro devolver un blob comprimido
    sin descomprimir para este dataset (ver justificacion)."""
    import kagglehub
    if dataset == "ibm_aml":
        return kagglehub.dataset_download(
            "ealtman2019/ibm-transactions-for-anti-money-laundering-aml", path=IBM_AML_FILE)
    elif dataset == "paysim":
        raw_dir = kagglehub.dataset_download("ealaxi/paysim1")
        return glob.glob(os.path.join(raw_dir, "*.csv"))[0]
    raise ValueError(f"Dataset desconocido: {dataset}")


> **Decision:** descargar unicamente `HI-Small_Trans.csv` de IBM AML con
> `kagglehub.dataset_download(handle, path="HI-Small_Trans.csv")`, en vez del
> dataset completo. Para PaySim se mantiene la descarga completa del dataset.
>
> **Justificacion:** IBM AML publica seis variantes (HI/LI x Small/Medium/Large)
> que suman ~7.6GB; el notebook solo usa `HI-Small_Trans.csv` (~475MB). Bajar
> todo el dataset consume minutos y espacio en disco innecesarios dentro del
> presupuesto de 30 minutos de Colab. Para PaySim no aplica la misma
> optimizacion: el dataset trae un unico CSV (sin ambiguedad que resolver), y
> al probar `path=<archivo>` en este dataset especifico, kagglehub devolvio un
> archivo mas pequeño que el original y no legible como CSV (aparentemente sin
> descomprimir), mientras que la descarga completa si funciona de forma
> confiable.
>
> **Alternativa descartada:** descargar el dataset completo de IBM AML con
> `kagglehub.dataset_download(handle)` y buscar el CSV correcto con un glob
> filtrando por substring (p. ej. `"HI-Small"`). Se descarta porque, ademas de
> ser mas lento, es ambiguo: ese substring coincide tanto con
> `HI-Small_Trans.csv` como con `HI-Small_accounts.csv`, y en una corrida real
> el glob selecciono el archivo equivocado (accounts en vez de transacciones),
> rompiendo el resto del pipeline en cascada.


In [ ]:
df = None

if not CFG["REBUILD_FROM_RAW"]:
    try:
        import requests
        if CFG["PROCESSED_URL"].startswith("TODO"):
            raise RuntimeError("PROCESSED_URL es un marcador TODO, no una URL real todavia.")
        r = requests.get(CFG["PROCESSED_URL"], timeout=15)
        r.raise_for_status()
        raise NotImplementedError("Descarga de .npz procesado no implementada en este marcador.")
    except Exception as e:
        warnings.warn(f"No se pudo usar la ruta de datos procesados ({e}). "
                       f"Cayendo a REBUILD_FROM_RAW=True.")
        CFG["REBUILD_FROM_RAW"] = True

if CFG["REBUILD_FROM_RAW"]:
    loader = load_ibm_aml if CFG["SEQUENCE_SOURCE"] == "ibm_aml" else load_paysim
    with Timer(f"descarga y carga de {CFG['SEQUENCE_SOURCE']} (raw)"):
        csv_path = get_raw_data_path(CFG["SEQUENCE_SOURCE"])
        print(f"Archivo cargado: {csv_path}")
        df = loader(csv_path)

print(df.shape)
df.head()


In [ ]:
# Diagnostico obligatorio antes de construir secuencias.
print("Transacciones por remitente (sender_id):")
print(df["sender_id"].value_counts().describe())
print()
print("Transacciones por destinatario (dest_id):")
print(df["dest_id"].value_counts().describe())


**Interpretacion del diagnostico:** si la media de transacciones por
`sender_id` es cercana a 1 (mediana = 1, percentil 75 = 1), significa que la
mayoria de remitentes en esta fuente aparecen en una sola transaccion, lo cual
hace imposible construir una secuencia temporal significativa por remitente.
Ese es exactamente el problema conocido de `nameOrig` en PaySim. Con IBM AML se
espera una distribucion con colas mas largas (remitentes recurrentes), lo que
confirma la decision tomada en la seccion 1 de usarlo como fuente principal.


In [ ]:
# PaySim se carga tambien, pero solo como validacion secundaria (no para
# construir secuencias por remitente): sirve para contrastar distribuciones de
# monto/tipo de transaccion y como chequeo cualitativo del pipeline de features.
with Timer("carga de PaySim (validacion secundaria)"):
    paysim_csv = get_raw_data_path("paysim")
    df_paysim = load_paysim(paysim_csv)

print("PaySim -- transacciones por remitente:")
print(df_paysim["sender_id"].value_counts().describe())


## 3. Ingenieria de features y construccion de secuencias

| Feature | Calculo | Por que |
|---|---|---|
| `log_amount` | `log1p(amount)` | colas pesadas en el monto |
| `delta_t` | `log1p(horas desde la transaccion anterior del mismo remitente)` | velocidad inusual de envios |
| `hour_sin`, `hour_cos` | codificacion ciclica de la hora | concentracion horaria |
| `tx_type` | one-hot | tipo de operacion |
| `is_new_dest` | 1 si el destino no aparecio antes en la secuencia del remitente | cambio abrupto de destinos |
| `dest_entropy` | entropia acumulada de destinos hasta ese punto | abanico (fan-out) de destinatarios |
| `threshold_ratio` | `amount / 10000` | fraccionamiento justo debajo del umbral regulatorio |


In [ ]:
def _is_new_dest(dest_series):
    seen = set()
    flags = []
    for d in dest_series:
        flags.append(0 if d in seen else 1)
        seen.add(d)
    return flags


def _dest_entropy(dest_series):
    counts = {}
    ent = []
    n = 0
    for d in dest_series:
        counts[d] = counts.get(d, 0) + 1
        n += 1
        probs = np.array(list(counts.values())) / n
        ent.append(float(-(probs * np.log(probs + 1e-12)).sum()))
    return ent


def build_features(df):
    df = df.sort_values(["sender_id", "timestamp"]).reset_index(drop=True).copy()

    df["log_amount"] = np.log1p(df["amount"].clip(lower=0))

    delta_h = df.groupby("sender_id")["timestamp"].diff().dt.total_seconds() / 3600.0
    df["delta_t_h"] = delta_h.fillna(0.0)
    df["delta_t"] = np.log1p(df["delta_t_h"].clip(lower=0))

    df["hour"] = df["timestamp"].dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["threshold_ratio"] = df["amount"] / 10000.0

    df["is_new_dest"] = df.groupby("sender_id")["dest_id"].transform(_is_new_dest)
    df["dest_entropy"] = df.groupby("sender_id")["dest_id"].transform(_dest_entropy)

    tx_dummies = pd.get_dummies(df["tx_type"], prefix="tx").astype(np.float32)
    df = pd.concat([df, tx_dummies], axis=1)

    return df, list(tx_dummies.columns)


NUMERIC_FEATURES = ["log_amount", "delta_t", "hour_sin", "hour_cos",
                     "is_new_dest", "dest_entropy", "threshold_ratio"]

with Timer("ingenieria de features"):
    df_feat, tx_cols = build_features(df)

feature_cols = NUMERIC_FEATURES + tx_cols
print(f"Features finales ({len(feature_cols)}): {feature_cols}")
df_feat[["sender_id", "timestamp"] + feature_cols].head()


In [ ]:
def build_sequences(df, feature_cols, cfg):
    lengths = df.groupby("sender_id").size()
    print(f"Remitentes totales: {len(lengths)}")

    valid_senders = lengths[lengths >= cfg["MIN_LEN"]].index
    dropped = len(lengths) - len(valid_senders)
    pos_senders = set(df.loc[df["label"] == 1, "sender_id"])
    dropped_senders = set(lengths.index) - set(valid_senders)
    dropped_pos = len(dropped_senders & pos_senders)
    print(f"Remitentes descartados por MIN_LEN={cfg['MIN_LEN']}: "
          f"{dropped} ({dropped / len(lengths):.2%})")
    print(f"Positivos perdidos por el filtro: {dropped_pos} de {len(pos_senders)} totales")

    p90 = int(np.percentile(lengths.values, 90))
    max_len = p90
    print(f"Percentil 90 de longitud de secuencia: {p90}. "
          f"MAX_LEN inicial en CFG: {cfg['MAX_LEN']}. MAX_LEN usado (derivado de datos): {max_len}")

    df_valid = df[df["sender_id"].isin(valid_senders)]

    if cfg["N_SENDERS"] and df_valid["sender_id"].nunique() > cfg["N_SENDERS"]:
        remaining_senders = df_valid["sender_id"].unique()
        pos_ids = [s for s in remaining_senders if s in pos_senders]
        neg_ids = [s for s in remaining_senders if s not in pos_senders]
        rng = np.random.RandomState(0)
        n_neg = max(cfg["N_SENDERS"] - len(pos_ids), 0)
        neg_sample = rng.choice(neg_ids, size=min(n_neg, len(neg_ids)), replace=False)
        keep = set(pos_ids) | set(neg_sample)
        df_valid = df_valid[df_valid["sender_id"].isin(keep)]
        print(f"Submuestreo estratificado a N_SENDERS={cfg['N_SENDERS']}: "
              f"{len(pos_ids)} positivos (todos) + {len(neg_sample)} negativos.")

    groups = df_valid.groupby("sender_id", sort=False)
    N = groups.ngroups
    F = len(feature_cols)
    X = np.zeros((N, max_len, F), dtype=np.float32)
    mask = np.zeros((N, max_len), dtype=bool)
    y = np.zeros((N,), dtype=np.int64)
    sender_ids = []
    kept_rows = []

    for i, (sid, g) in enumerate(groups):
        g = g.sort_values("timestamp").tail(max_len)
        L = len(g)
        X[i, max_len - L:, :] = g[feature_cols].to_numpy()
        mask[i, max_len - L:] = True
        y[i] = int((g["label"] == 1).any())
        sender_ids.append(sid)
        kept_rows.append(g)

    sender_ids = np.array(sender_ids, dtype=object)
    df_kept = pd.concat(kept_rows, axis=0)
    return X, mask, y, sender_ids, max_len, df_kept


with Timer("construccion de secuencias"):
    X, mask, y, sender_ids, max_len, df_valid = build_sequences(df_feat, feature_cols, CFG)

print(f"X: {X.shape}, mask: {mask.shape}, y: {y.shape}, positivos: {y.sum()} ({y.mean():.4%})")


> **Decision:** truncar cada secuencia a las ultimas `MAX_LEN` transacciones
> (padding a la izquierda) y descartar remitentes con menos de `MIN_LEN=5`
> transacciones.
>
> **Justificacion:** `MAX_LEN` se deriva del percentil 90 de la distribucion de
> longitudes real (impreso arriba), no de un valor arbitrario. Truncar por la
> derecha (quedarse con las transacciones mas recientes) preserva la senal mas
> relevante para un sistema de alertas, que opera sobre comportamiento reciente.
>
> **Alternativa descartada:** usar la secuencia completa sin truncar. Se
> descarta porque produce tensores de forma variable extrema y dispara el
> costo computacional sin aportar señal adicional para remitentes con miles de
> transacciones.


> **Nota honesta sobre este calculo:** el percentil 90 se calcula sobre la
> longitud de **todos** los remitentes (incluyendo los ~74% que se descartan
> despues por `MIN_LEN=5`), no solo sobre los que sobreviven el filtro. Eso
> arrastra el percentil hacia abajo: en la corrida real dio `MAX_LEN=29`,
> cuando calcularlo solo sobre los remitentes validos (>=5 transacciones)
> habria dado un valor bastante mayor (~64), capturando mas historial por
> secuencia. Se documenta como limitacion conocida en vez de corregirla y
> volver a correr todo el pipeline (~20-30 min); `MAX_LEN=29` sigue siendo
> data-driven y no arbitrario, solo que sobre una poblacion menos
> representativa de lo ideal.

## 4. Splits y verificacion de fuga

In [ ]:
def split_by_sender(sender_ids, y, train_frac=0.70, val_frac=0.15, seed=0):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(sender_ids))
    pos_idx = idx[y == 1].copy()
    neg_idx = idx[y == 0].copy()
    rng.shuffle(pos_idx)
    rng.shuffle(neg_idx)

    def split_group(arr):
        n = len(arr)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)
        return arr[:n_train], arr[n_train:n_train + n_val], arr[n_train + n_val:]

    pos_tr, pos_va, pos_te = split_group(pos_idx)
    neg_tr, neg_va, neg_te = split_group(neg_idx)

    train_idx = np.concatenate([pos_tr, neg_tr])
    val_idx = np.concatenate([pos_va, neg_va])
    test_idx = np.concatenate([pos_te, neg_te])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


train_idx, val_idx, test_idx = split_by_sender(sender_ids, y, seed=CFG["SEEDS"][0])

train_senders = set(sender_ids[train_idx])
val_senders = set(sender_ids[val_idx])
test_senders = set(sender_ids[test_idx])

assert train_senders.isdisjoint(val_senders), "Fuga: remitentes compartidos entre train y val"
assert train_senders.isdisjoint(test_senders), "Fuga: remitentes compartidos entre train y test"
assert val_senders.isdisjoint(test_senders), "Fuga: remitentes compartidos entre val y test"
print("Assert de no-solapamiento de remitentes entre splits: OK")

print(f"train: {len(train_idx)} ({y[train_idx].mean():.4%} positivos)")
print(f"val:   {len(val_idx)} ({y[val_idx].mean():.4%} positivos)")
print(f"test:  {len(test_idx)} ({y[test_idx].mean():.4%} positivos)")


In [ ]:
# Corte temporal adicional: se compara la ultima transaccion de cada remitente
# por split. Si test tuviera remitentes sistematicamente mas antiguos que train,
# el modelo estaria aprendiendo a "predecir hacia atras en el tiempo", lo cual
# no reflejaria un despliegue real (donde el modelo ve el pasado y predice sobre
# comportamiento futuro).
sender_last_ts = df_valid.groupby("sender_id")["timestamp"].max()
for name, s_idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    ts = sender_last_ts.loc[sender_ids[s_idx]]
    print(f"{name}: ultima transaccion promedio = {ts.mean()}, min = {ts.min()}, max = {ts.max()}")

# Riesgo de fuga temporal: con un split puramente aleatorio por remitente (como
# el usado arriba) es posible que un remitente de test tenga transacciones mas
# antiguas que uno de train, lo que en un despliegue real no ocurriria (el
# modelo nunca "ve el futuro" de un remitente que ya proceso). Este proyecto
# usa split aleatorio estratificado por ser mas simple de balancear por clase,
# pero de llevarse a produccion se recomienda un split temporal estricto
# (entrenar con remesas anteriores a una fecha de corte, evaluar con las
# posteriores) para eliminar este riesgo por completo.


In [ ]:
# Normalizacion ajustada SOLO sobre el split de entrenamiento, para evitar fuga
# de estadisticas de val/test hacia el entrenamiento.
train_valid_mask = mask[train_idx]
train_valid_values = X[train_idx][train_valid_mask]

feat_mean = train_valid_values.mean(axis=0)
feat_std = train_valid_values.std(axis=0) + 1e-6


def normalize(X, mask, mean, std):
    Xn = X.copy()
    Xn[mask] = (X[mask] - mean) / std
    return Xn


X_norm = normalize(X, mask, feat_mean, feat_std)
norm_stats = {"mean": feat_mean.tolist(), "std": feat_std.tolist()}
print("Normalizacion ajustada sobre train. Ejemplo de medias:", feat_mean[:3])


## 5. Visualizaciones exigidas por el enunciado (Componente 1)

In [ ]:
import matplotlib.pyplot as plt

lengths_full = df_feat.groupby("sender_id").size()

plt.figure(figsize=(8, 4))
plt.hist(lengths_full.values, bins=60)
plt.xlabel("Transacciones por remitente")
plt.ylabel("Frecuencia")
plt.title("Distribucion de longitud de secuencia por remitente")
plt.axvline(CFG["MIN_LEN"], color="orange", linestyle="--", label=f"MIN_LEN={CFG['MIN_LEN']}")
plt.axvline(max_len, color="red", linestyle="--", label=f"MAX_LEN={max_len} (p90)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
rows = []
for name, idx_ in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    yy = y[idx_]
    rows.append({
        "split": name, "n": len(yy),
        "positivos": int(yy.sum()), "negativos": int((yy == 0).sum()),
        "prop_positivos": float(yy.mean()),
    })
split_table = pd.DataFrame(rows)
split_table


In [ ]:
# Tres secuencias normales vs. tres sospechosas, como heatmap de features a lo
# largo del tiempo (eje x = indice de transaccion dentro de la secuencia).
rng = np.random.RandomState(0)
test_pos = np.where(y[test_idx] == 1)[0]
test_neg = np.where(y[test_idx] == 0)[0]
pos_examples = rng.choice(test_pos, size=min(3, len(test_pos)), replace=False)
neg_examples = rng.choice(test_neg, size=min(3, len(test_neg)), replace=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for col, i in enumerate(neg_examples):
    g_idx = test_idx[i]
    valid = mask[g_idx]
    axes[0, col].imshow(X_norm[g_idx][valid].T, aspect="auto", cmap="viridis")
    axes[0, col].set_title(f"Normal -- {sender_ids[g_idx]}")
    axes[0, col].set_xlabel("indice de transaccion")
for col, i in enumerate(pos_examples):
    g_idx = test_idx[i]
    valid = mask[g_idx]
    axes[1, col].imshow(X_norm[g_idx][valid].T, aspect="auto", cmap="magma")
    axes[1, col].set_title(f"Sospechoso -- {sender_ids[g_idx]}")
    axes[1, col].set_xlabel("indice de transaccion")
axes[0, 0].set_ylabel("features (fila = feature)")
axes[1, 0].set_ylabel("features (fila = feature)")
plt.tight_layout()
plt.show()


## 6. Etapa A -- aprendizaje de la normalidad

Autoencoder secuencial con atencion. La atencion no es decorativa: es el
insumo del mapa de calor del MVP y del analisis de interpretabilidad de la
seccion 9.


In [ ]:
class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, X, mask, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.mask[i], self.y[i]


class AttnPool(nn.Module):
    """Atencion aditiva (Bahdanau). Devuelve (context, alpha).
    alpha: (B, L), softmax enmascarado, suma 1 sobre posiciones validas."""
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, 1, bias=False)

    def forward(self, h, mask):
        scores = self.v(torch.tanh(self.W(h))).squeeze(-1)
        scores = scores.masked_fill(~mask, float("-inf"))
        alpha = torch.softmax(scores, dim=1)
        alpha = torch.nan_to_num(alpha, nan=0.0)
        context = (alpha.unsqueeze(-1) * h).sum(dim=1)
        return context, alpha


class SeqEncoder(nn.Module):
    """GRU bidireccional (HIDDEN) -> AttnPool -> proyeccion a LATENT.
    forward(x, mask) -> (z: (B, LATENT), alpha: (B, L))"""
    def __init__(self, n_features, hidden, latent):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, batch_first=True, bidirectional=True)
        self.attn = AttnPool(hidden * 2)
        self.proj = nn.Linear(hidden * 2, latent)

    def forward(self, x, mask):
        lengths = mask.sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        h_packed, _ = self.gru(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h_packed, batch_first=True, total_length=x.size(1))
        context, alpha = self.attn(h, mask)
        z = self.proj(context)
        return z, alpha


class SeqDecoder(nn.Module):
    """Decoder autoregresivo con teacher forcing. En cada paso t la entrada es
    concat([z, x_{t-1}]) (x_{-1} = token de inicio aprendido), en vez de
    repetir z como unica entrada constante en los L pasos.
    forward(z, x) -> x_hat: (B, L, F)"""
    def __init__(self, latent, hidden, n_features):
        super().__init__()
        self.n_features = n_features
        self.gru = nn.GRU(latent + n_features, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_features)
        self.start_token = nn.Parameter(torch.zeros(n_features))

    def forward(self, z, x):
        B, L, _ = x.shape
        start = self.start_token.view(1, 1, -1).expand(B, 1, -1)
        prev = torch.cat([start, x[:, :-1, :]], dim=1)  # x desplazado un paso (teacher forcing)
        z_rep = z.unsqueeze(1).repeat(1, L, 1)
        dec_input = torch.cat([z_rep, prev], dim=-1)
        h, _ = self.gru(dec_input)
        return self.out(h)


class SeqAutoencoder(nn.Module):
    """forward(x, mask) -> (x_hat, z, alpha)"""
    def __init__(self, n_features, hidden, latent):
        super().__init__()
        self.encoder = SeqEncoder(n_features, hidden, latent)
        self.decoder = SeqDecoder(latent, hidden, n_features)

    def forward(self, x, mask):
        z, alpha = self.encoder(x, mask)
        x_hat = self.decoder(z, x)
        return x_hat, z, alpha


def masked_mse(x_hat, x, mask):
    """MSE enmascarado por secuencia: se divide entre el numero de posiciones
    validas, no entre L, para no penalizar secuencias cortas de mas."""
    diff2 = (x_hat - x) ** 2
    diff2 = diff2 * mask.unsqueeze(-1)
    denom = mask.sum(dim=1).clamp(min=1) * x.size(-1)
    return diff2.sum(dim=(1, 2)) / denom


> **Decision:** el decoder de la Etapa A es autoregresivo con teacher
> forcing (recibe `[z, x_{t-1}]` en cada paso) en vez de repetir `z` como
> entrada constante en los `L` pasos.
>
> **Justificacion:** en una primera version del notebook, el decoder recibia
> unicamente `z` repetido en cada paso. Con eso, la unica forma que tenia el
> decoder de distinguir el paso `t=0` del `t=30` era la deriva de su propio
> estado oculto recurrente a partir de una entrada identica, lo cual resulto
> ser muy poca señal: en una corrida real con datos de IBM AML, el MSE de
> validacion apenas bajo de 0.81 a 0.74 en 15 epocas (con features
> normalizadas a media 0 / varianza 1, un MSE de referencia trivial --
> "predecir siempre la media" -- es 1.0), es decir, el autoencoder aprendia
> una reconstruccion casi trivial. Alimentar la transaccion anterior real
> (`x_{t-1}`) en cada paso le da al decoder señal concreta sobre la dinamica
> de la secuencia, y el error de reconstruccion se vuelve sensible a
> desviaciones genuinas del patron esperado dado `z`, no solo a la falta de
> informacion temporal.
>
> **Alternativa descartada:** la version con `z` repetido sin informacion
> adicional (ver justificacion). Tambien se considero un decoder con atencion
> propia sobre los estados del encoder, pero se descarta porque diluiria el
> proposito del cuello de botella (`z` como resumen comprimido del
> comportamiento, que es lo que hace que el score de anomalia sea
> interpretable como "que tan bien resume z a esta secuencia").


In [ ]:
def train_stage_a(X, mask, y, train_idx, val_idx, cfg, seed):
    set_seed(seed)
    device = cfg["DEVICE"]

    neg_train_idx = train_idx[y[train_idx] == 0]
    print(f"Etapa A (seed={seed}) -- remitentes negativos de entrenamiento: "
          f"{len(neg_train_idx)} de {len(train_idx)} totales en train")

    train_ds = SeqDataset(X[neg_train_idx], mask[neg_train_idx], y[neg_train_idx])
    val_ds = SeqDataset(X[val_idx], mask[val_idx], y[val_idx])
    g = torch.Generator().manual_seed(seed)
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=cfg["BATCH"], shuffle=True, generator=g)
    val_dl = torch.utils.data.DataLoader(val_ds, batch_size=cfg["BATCH"], shuffle=False)

    n_features = X.shape[-1]
    model = SeqAutoencoder(n_features, cfg["HIDDEN"], cfg["LATENT"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = {"train": [], "val": []}
    for epoch in range(cfg["EPOCHS_A"]):
        model.train()
        tr_losses = []
        for xb, mb, _ in train_dl:
            xb, mb = xb.to(device), mb.to(device)
            x_hat, z, alpha = model(xb, mb)
            loss = masked_mse(x_hat, xb, mb).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            tr_losses.append(loss.item())

        model.eval()
        va_losses = []
        with torch.no_grad():
            for xb, mb, _ in val_dl:
                xb, mb = xb.to(device), mb.to(device)
                x_hat, z, alpha = model(xb, mb)
                va_losses.append(masked_mse(x_hat, xb, mb).mean().item())

        history["train"].append(float(np.mean(tr_losses)))
        history["val"].append(float(np.mean(va_losses)))
        print(f"[Etapa A][seed={seed}] epoch {epoch + 1}/{cfg['EPOCHS_A']} "
              f"train={history['train'][-1]:.4f} val={history['val'][-1]:.4f}")

    return model, history


@torch.no_grad()
def compute_recon_scores(model, X, mask, cfg):
    model.eval()
    ds = SeqDataset(X, mask, np.zeros(len(X)))
    dl = torch.utils.data.DataLoader(ds, batch_size=cfg["BATCH"], shuffle=False)
    scores, alphas = [], []
    for xb, mb, _ in dl:
        xb, mb = xb.to(cfg["DEVICE"]), mb.to(cfg["DEVICE"])
        x_hat, z, alpha = model(xb, mb)
        scores.append(masked_mse(x_hat, xb, mb).cpu().numpy())
        alphas.append(alpha.cpu().numpy())
    return np.concatenate(scores), np.concatenate(alphas)


In [ ]:
with Timer(f"Etapa A -- entrenamiento (seed={CFG['SEEDS'][0]})"):
    ae_model, ae_history = train_stage_a(X_norm, mask, y, train_idx, val_idx, CFG, CFG["SEEDS"][0])

plt.figure(figsize=(6, 4))
plt.plot(ae_history["train"], label="train")
plt.plot(ae_history["val"], label="val")
plt.xlabel("epoch")
plt.ylabel("MSE enmascarado")
plt.title("Curva de perdida -- Etapa A")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve, fbeta_score

recon_val, _ = compute_recon_scores(ae_model, X_norm[val_idx], mask[val_idx], CFG)
y_val = y[val_idx]


def find_threshold_f2(scores_val, y_val):
    prec, rec, thr = precision_recall_curve(y_val, scores_val)
    f2 = (5 * prec * rec) / (4 * prec + rec + 1e-12)
    f2 = f2[:-1]
    if len(thr) == 0:
        return float(np.median(scores_val)), 0.0
    best_i = int(np.nanargmax(f2))
    return float(thr[best_i]), float(f2[best_i])


def find_threshold_alert_rate(scores_val, alert_rate):
    return float(np.quantile(scores_val, 1 - alert_rate))


thr_f2, f2_at_thr = find_threshold_f2(recon_val, y_val)
thr_alert = find_threshold_alert_rate(recon_val, CFG["ALERT_RATE"])

threshold_table = pd.DataFrame([
    {"criterio": "maximiza F2 en validacion", "umbral": thr_f2, "F2": f2_at_thr},
    {"criterio": f"tasa de alertas = {CFG['ALERT_RATE']:.1%}", "umbral": thr_alert, "F2": None},
])
threshold_table


In [ ]:
# Umbral oficial: se elige el basado en ALERT_RATE.
official_threshold = thr_alert
print(f"Umbral oficial de Etapa A: {official_threshold:.4f} (basado en ALERT_RATE={CFG['ALERT_RATE']:.1%})")


> **Decision:** usar el umbral basado en `ALERT_RATE` (tasa de alertas =
> capacidad operativa del equipo de cumplimiento) como umbral oficial de la
> Etapa A, en vez del que maximiza F2.
>
> **Justificacion:** un umbral que maximiza F2 en validacion puede generar mas
> alertas de las que un equipo de cumplimiento puede revisar manualmente. El
> problema de negocio (seccion de contexto del enunciado) es precisamente que
> los sistemas actuales generan demasiadas alertas para revisar. Un umbral
> atado a la capacidad operativa real es la decision que un banco desplegaria
> en produccion; el umbral F2 se reporta igual como referencia de que tan lejos
> esta del optimo estadistico.
>
> **Alternativa descartada:** usar el umbral que maximiza F2 como oficial. Se
> descarta porque optimiza una metrica estadistica sin considerar la
> restriccion operativa, que es el cuello de botella real descrito en el
> contexto del problema.


## 7. Etapa B -- transfer learning y perdida

El clasificador reutiliza el encoder de la Etapa A (transfer learning con
congelamiento inicial y luego fine-tuning con LR discriminativo), en lugar de
aprender desde cero.


In [ ]:
class DetectorHead(nn.Module):
    def __init__(self, latent):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent + 1, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, z, recon_error_norm):
        inp = torch.cat([z, recon_error_norm.unsqueeze(-1)], dim=-1)
        return self.net(inp).squeeze(-1)


class Detector(nn.Module):
    """Encoder inicializado desde Etapa A + cabeza MLP.
    Entrada a la cabeza: concat([z, recon_error_normalizado]) -> (LATENT + 1)
    forward(x, mask) -> (logit, alpha, recon_error)"""
    def __init__(self, encoder, decoder, recon_mean, recon_std, use_recon_feature=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.use_recon_feature = use_recon_feature
        self.head = DetectorHead(cfg_latent(encoder))
        self.register_buffer("recon_mean", torch.tensor(float(recon_mean)))
        self.register_buffer("recon_std", torch.tensor(float(recon_std) + 1e-6))

    def forward(self, x, mask):
        z, alpha = self.encoder(x, mask)
        x_hat = self.decoder(z, x)
        recon_error = masked_mse(x_hat, x, mask)
        if self.use_recon_feature:
            recon_norm = (recon_error - self.recon_mean) / self.recon_std
        else:
            recon_norm = torch.zeros_like(recon_error)
        logit = self.head(z, recon_norm)
        return logit, alpha, recon_error


def cfg_latent(encoder):
    return encoder.proj.out_features


def focal_loss(logits, targets, gamma=2.0, alpha=0.25):
    """Focal loss implementada a mano (no se importa de ninguna libreria).
    gamma controla cuanto se reduce el peso de ejemplos faciles (ya bien
    clasificados); alpha balancea la clase positiva minoritaria."""
    p = torch.sigmoid(logits)
    ce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t) ** gamma * ce
    return loss.mean()


> **Decision:** `gamma=2.0`, `alpha=0.25` para la focal loss, e implementarla
> a mano en vez de usar una libreria externa.
>
> **Justificacion:** el desbalance de clases en AML es extremo (los casos
> positivos son una fraccion minima). La focal loss reduce el peso de los
> ejemplos negativos faciles (la gran mayoria) y concentra el gradiente en los
> ejemplos dificiles y en la clase minoritaria, que es exactamente lo que
> `alpha=0.25` corrige adicionalmente. `gamma=2.0` es el valor original del
> paper de Focal Loss (Lin et al., 2017) y funciona como punto de partida
> razonable antes de afinar con validacion.
>
> **Alternativa descartada:** binary cross-entropy simple con pesos por clase
> (`pos_weight`). Se descarta porque solo reescala el gradiente por clase, sin
> reducir explicitamente la contribucion de los ejemplos faciles ya bien
> clasificados, que es el problema dominante cuando el desbalance es tan
> extremo.

> **Decision:** el error de reconstruccion de la Etapa A entra como feature
> adicional concatenada a `z` en la cabeza del clasificador, en vez de
> promediarse por separado con la probabilidad de la Etapa B.
>
> **Justificacion:** el error de reconstruccion y la probabilidad de la cabeza
> viven en escalas completamente distintas (uno es un MSE sin cota superior
> natural, el otro es una probabilidad en [0,1]). Promediarlos directamente
> requeriria una calibracion arbitraria. Dejar que la cabeza aprenda el peso
> optimo de esa senal junto con la representacion latente es mas robusto y se
> ajusta durante el entrenamiento supervisado.
>
> **Alternativa descartada:** promediar `anomaly_score_a` y `prob_b` con pesos
> fijos (p. ej. 0.5/0.5). Se descarta por la razon de escala mencionada arriba
> y porque el peso relativo entre ambas senales deberia depender de que tan
> confiable es cada una para cada caso, algo que un promedio fijo no captura.


In [ ]:
def train_stage_b(base_encoder, base_decoder, X, mask, y, train_idx, val_idx, cfg, seed,
                   from_scratch=False, freeze_epochs=0, use_recon_feature=True,
                   recon_mean=0.0, recon_std=1.0):
    set_seed(seed)
    device = cfg["DEVICE"]
    n_features = X.shape[-1]

    if from_scratch:
        encoder = SeqEncoder(n_features, cfg["HIDDEN"], cfg["LATENT"])
        decoder = SeqDecoder(cfg["LATENT"], cfg["HIDDEN"], n_features)
    else:
        encoder = copy.deepcopy(base_encoder)
        decoder = copy.deepcopy(base_decoder)

    model = Detector(encoder, decoder, recon_mean, recon_std, use_recon_feature).to(device)
    for p in model.decoder.parameters():
        p.requires_grad_(False)  # el decoder solo se usa para calcular recon_error, no se reentrena

    train_ds = SeqDataset(X[train_idx], mask[train_idx], y[train_idx])
    val_ds = SeqDataset(X[val_idx], mask[val_idx], y[val_idx])
    g = torch.Generator().manual_seed(seed)
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=cfg["BATCH"], shuffle=True, generator=g)
    val_dl = torch.utils.data.DataLoader(val_ds, batch_size=cfg["BATCH"], shuffle=False)

    for epoch in range(cfg["EPOCHS_B"]):
        freeze_now = (epoch < freeze_epochs) and not from_scratch
        for p in model.encoder.parameters():
            p.requires_grad_(not freeze_now)

        param_groups = [{"params": model.head.parameters(), "lr": cfg["LR_HEAD"]}]
        if not freeze_now:
            enc_params = [p for p in model.encoder.parameters() if p.requires_grad]
            param_groups.append({"params": enc_params, "lr": cfg["LR_ENCODER"]})
        opt = torch.optim.Adam(param_groups)

        model.train()
        tr_losses = []
        for xb, mb, yb in train_dl:
            xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
            logit, alpha, recon_error = model(xb, mb)
            loss = focal_loss(logit, yb, gamma=cfg["FOCAL_GAMMA"], alpha=cfg["FOCAL_ALPHA"])
            opt.zero_grad()
            loss.backward()
            opt.step()
            tr_losses.append(loss.item())

        model.eval()
        va_losses = []
        with torch.no_grad():
            for xb, mb, yb in val_dl:
                xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
                logit, alpha, recon_error = model(xb, mb)
                va_losses.append(focal_loss(logit, yb, gamma=cfg["FOCAL_GAMMA"], alpha=cfg["FOCAL_ALPHA"]).item())

        estado = "congelado" if freeze_now else "fine-tuning"
        print(f"[Etapa B][seed={seed}] epoch {epoch + 1}/{cfg['EPOCHS_B']} ({estado}) "
              f"train={np.mean(tr_losses):.4f} val={np.mean(va_losses):.4f}")

    return model


@torch.no_grad()
def evaluate_detector(model, X, mask, y, cfg):
    model.eval()
    ds = SeqDataset(X, mask, y)
    dl = torch.utils.data.DataLoader(ds, batch_size=cfg["BATCH"], shuffle=False)
    probs, alphas, recon_errors, labels = [], [], [], []
    for xb, mb, yb in dl:
        xb, mb = xb.to(cfg["DEVICE"]), mb.to(cfg["DEVICE"])
        logit, alpha, recon_error = model(xb, mb)
        probs.append(torch.sigmoid(logit).cpu().numpy())
        alphas.append(alpha.cpu().numpy())
        recon_errors.append(recon_error.cpu().numpy())
        labels.append(yb.numpy())
    return (np.concatenate(probs), np.concatenate(alphas),
            np.concatenate(recon_errors), np.concatenate(labels))


from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score


def compute_metrics(probs, labels, alert_rate):
    auprc = average_precision_score(labels, probs)
    roc_auc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan")
    thr = np.quantile(probs, 1 - alert_rate)
    preds = (probs >= thr).astype(int)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f2 = fbeta_score(labels, preds, beta=2, zero_division=0)
    return dict(auprc=auprc, roc_auc=roc_auc, precision_at_alert=prec, recall_at_alert=rec, f2=f2)
